# CEREBRO: Decision Reversal Pattern Detector
**Kolkata IEM Hackathon | Problem Statement 3**  
*Deterministic Behavioral Intelligence & Multi-Stage Choice Oscillations*

---

## 1. Problem Statement
In online subscription checkout funnels, users frequently change their minds—switching subscription plans, toggling add-ons, or reconsidering payment methods. Conventional funnel telemetry only measures coarse linear drop-offs, missing the critical psychological micro-frictions, hesitation intervals, and circular oscillations that precede abandonment or purchase regret.

**Goal:** Given a sequence of timestamped user choices recorded as events, detect when a user repeatedly changes a previous choice, identify the funnel stages where reversals concentrate, classify the user's behavioural pattern using strict rule-based priority, explain the reasoning in plain English, and provide interactive Plotly visualisations.

---

## 2. Formal Domain Definitions

| Term | Formal Definition |
| :--- | :--- |
| **Decision Event** | A user selection tuple $(u, d, c, s, t)$ where user $u$ chooses option $c$ for decision $d$ at funnel stage $s$ and timestamp $t$. |
| **Reversal** | An event where the choice differs from the immediately preceding choice for the same $(u, d)$ pair: $c_t \neq c_{t-1}$. |
| **Flip-Back** | A reversal where the new choice matches *any* earlier choice recorded for that $(u, d)$ pair: $c_t \in \{c_1, \dots, c_{t-1}\}$. |
| **Correction** | Exactly one reversal on a decision, made within $\Delta t \le \text{CORRECTION\_MAX\_SECONDS}$ (default 60 s) with no flip-back. |
| **Reversal Rate** | The ratio $\frac{\text{total\_reversals}}{\text{number\_of\_distinct\_decisions\_made}}$. |
| **Funnel Stages** | Ordered linear checkout milestones: `Browsing → Selection → Review → Confirmation → Post-commit`. |

---

## 3. End-to-End Pipeline Architecture

```
[Raw Event Log / CSV]
         │
         ▼
[1. Loader & Validator]  ───▶  Checks schema, parses ISO-8601 timestamps, validates stages, sorts
         │
         ▼
[2. Reversal Detector]   ───▶  Computes Δt, detects state transitions & flip-backs (from -> to)
         │
         ▼
[3. Metric Aggregator]   ───▶  Extracts per-user KPIs (rates, hesitation, concentration shares)
         │
         ▼
[4. Priority Classifier] ───▶  Evaluates 6 mutually exclusive behavioral rules
         │
         ▼
[5. Explainability]      ───▶  Generates deterministic, template-based natural evidence sentences
         │
         ▼
[6. Interactive Visuals] ───▶  Plotly Timeline, Stage Distribution, and Sankey Decision Flow
```


## Section 1: Setup & Central Configuration
This section installs dependencies (if needed), imports all required libraries, fixes global random seeds, and establishes a single immutable `CONFIG` dictionary containing every operational threshold.

In [1]:
# Optional dependency installation (uncomment if running in fresh environment)
# !pip install -q pandas numpy plotly scikit-learn ipywidgets hmmlearn nbformat

In [2]:
import os
import io
import json
import math
import random
import pathlib
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Global Random Seed for Deterministic Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Central Configuration Dictionary
CONFIG: Dict[str, Any] = {
    "SEED": 42,
    # Correction threshold: max seconds for a single quick change
    "CORRECTION_MAX_SECONDS": 60,
    # Flip-flopper thresholds: repeated oscillation on a single decision
    "FLIP_FLOPPER_MIN_REVERSALS_ON_ONE_DECISION": 3,
    "FLIP_FLOPPER_MIN_FLIP_BACKS": 2,
    "FLIP_FLOPPER_MIN_SHARE": 0.60,
    # Late reverser threshold: share of reversals at Confirmation / Post-commit
    "LATE_REVERSER_MIN_LATE_SHARE": 0.60,
    # Chronic indecision thresholds: broad multi-decision churn
    "CHRONIC_INDECISIVE_MIN_REVERSAL_RATE": 0.75,
    "CHRONIC_INDECISIVE_MIN_DECISIONS": 2,
    "CHRONIC_INDECISIVE_MAX_SHARE": 0.60,
    # Funnel and domain definitions
    "STAGE_ORDER": ["Browsing", "Selection", "Review", "Confirmation", "Post-commit"],
    "LATE_STAGES": ["Confirmation", "Post-commit"],
    "DECISION_TYPES": ["plan", "add_on", "payment_method"],
    "CHOICE_SPACE": {
        "plan": ["basic", "pro", "enterprise"],
        "add_on": ["none", "analytics", "priority_support", "all_inclusive"],
        "payment_method": ["credit_card", "debit_card", "paypal", "crypto"],
    },
    "CSV_COLUMNS": ["user_id", "decision_id", "timestamp", "stage", "choice"],
    "TIMESTAMP_COL": "timestamp",
    "ENABLE_AFFECTIVE_LAYER": False,  # Optional behavioural uncertainty HMM layer
}

print(f"✅ Setup complete. Global SEED = {CONFIG['SEED']}")
print(f"✅ Canonical Funnel: {' -> '.join(CONFIG['STAGE_ORDER'])}")
print(f"✅ Affective Layer Enabled: {CONFIG['ENABLE_AFFECTIVE_LAYER']}")

✅ Setup complete. Global SEED = 42
✅ Canonical Funnel: Browsing -> Selection -> Review -> Confirmation -> Post-commit
✅ Affective Layer Enabled: False


## Section 2: Synthetic Data Generation
We generate a balanced benchmark of 200 synthetic users across 5 distinct personas (40 users per persona) with realistic timestamp intervals, multi-stage progressions, and behavioral oscillation patterns. The generated data is saved to `data/sample_events.csv` and `data/ground_truth.csv`.

In [3]:
def _decision_id(user_id: str, decision_type: str) -> str:
    """Stable decision identifier for a (user, decision_type) pair."""
    return f"{user_id}_{decision_type}"

def _other_choice(current: str, dtype: str, config: Dict[str, Any] = CONFIG) -> str:
    """Pick a different choice from the same decision type."""
    options = [c for c in config["CHOICE_SPACE"][dtype] if c != current]
    return random.choice(options)

def _earlier_choice(history: List[str], current: str, dtype: str, config: Dict[str, Any] = CONFIG) -> str:
    """Return a choice that was used before in history AND differs from current."""
    earlier = [c for c in history if c != current]
    if earlier:
        return random.choice(earlier)
    return _other_choice(current, dtype, config)

def _make_event(user_id: str, dtype: str, stage: str, choice: str, ts: datetime) -> dict:
    """Build a single event row dictionary."""
    return {
        "user_id": user_id,
        "decision_id": _decision_id(user_id, dtype),
        "timestamp": ts.isoformat(timespec="seconds"),
        "stage": stage,
        "choice": choice,
    }

def _build_decisive(user_id: str, session_start: datetime, config: Dict[str, Any] = CONFIG) -> List[dict]:
    events = []
    ts = session_start
    stage_slots = random.sample(config["STAGE_ORDER"][:4], k=3)
    for dtype, stage in zip(config["DECISION_TYPES"], sorted(stage_slots, key=config["STAGE_ORDER"].index)):
        choice = random.choice(config["CHOICE_SPACE"][dtype])
        ts += timedelta(seconds=random.randint(30, 120))
        events.append(_make_event(user_id, dtype, stage, choice, ts))
    return events

def _build_single_corrector(user_id: str, session_start: datetime, config: Dict[str, Any] = CONFIG) -> List[dict]:
    events = []
    ts = session_start
    corrected_dtype = random.choice(config["DECISION_TYPES"])
    for dtype in config["DECISION_TYPES"]:
        stage = random.choice(config["STAGE_ORDER"][:3])
        choice = random.choice(config["CHOICE_SPACE"][dtype])
        ts += timedelta(seconds=random.randint(30, 90))
        events.append(_make_event(user_id, dtype, stage, choice, ts))
        if dtype == corrected_dtype:
            gap = random.randint(5, config["CORRECTION_MAX_SECONDS"])
            ts += timedelta(seconds=gap)
            new_choice = _other_choice(choice, dtype, config)
            events.append(_make_event(user_id, dtype, stage, new_choice, ts))
    return events

def _build_flip_flopper(user_id: str, session_start: datetime, config: Dict[str, Any] = CONFIG) -> List[dict]:
    events = []
    ts = session_start
    flop_dtype = random.choice(config["DECISION_TYPES"])
    history: List[str] = []
    initial_stage = random.choice(["Browsing", "Selection"])
    first_choice = random.choice(config["CHOICE_SPACE"][flop_dtype])
    ts += timedelta(seconds=random.randint(15, 60))
    events.append(_make_event(user_id, flop_dtype, initial_stage, first_choice, ts))
    history.append(first_choice)
    current = first_choice

    n_changes = random.randint(3, 5)
    flip_back_quota = 2
    stage_progression = iter(config["STAGE_ORDER"][1:])
    next_stage = next(stage_progression, "Review")

    for i in range(n_changes):
        ts += timedelta(seconds=random.randint(30, 180))
        if flip_back_quota > 0 and len(history) >= 2:
            new_choice = _earlier_choice(history, current, flop_dtype, config)
            flip_back_quota -= 1
        else:
            new_choice = _other_choice(current, flop_dtype, config)

        if random.random() < 0.6:
            next_stage = next(stage_progression, config["STAGE_ORDER"][-1])

        events.append(_make_event(user_id, flop_dtype, next_stage, new_choice, ts))
        history.append(new_choice)
        current = new_choice

    for dtype in config["DECISION_TYPES"]:
        if dtype == flop_dtype:
            continue
        stage = random.choice(config["STAGE_ORDER"][:2])
        choice = random.choice(config["CHOICE_SPACE"][dtype])
        ts += timedelta(seconds=random.randint(20, 60))
        events.append(_make_event(user_id, dtype, stage, choice, ts))
        if random.random() < 0.3:
            ts += timedelta(seconds=random.randint(60, 200))
            events.append(_make_event(user_id, dtype, "Review", _other_choice(choice, dtype, config), ts))
    return events

def _build_late_reverser(user_id: str, session_start: datetime, config: Dict[str, Any] = CONFIG) -> List[dict]:
    events = []
    ts = session_start
    n_late = random.randint(1, 2)
    late_dtypes = random.sample(config["DECISION_TYPES"], k=n_late)

    initial_choices: Dict[str, str] = {}
    for dtype in config["DECISION_TYPES"]:
        stage = random.choice(config["STAGE_ORDER"][:3])
        choice = random.choice(config["CHOICE_SPACE"][dtype])
        ts += timedelta(seconds=random.randint(30, 120))
        events.append(_make_event(user_id, dtype, stage, choice, ts))
        initial_choices[dtype] = choice

    ts += timedelta(seconds=random.randint(300, 900))
    for dtype in late_dtypes:
        late_stage = random.choice(config["LATE_STAGES"])
        new_choice = _other_choice(initial_choices[dtype], dtype, config)
        ts += timedelta(seconds=random.randint(30, 120))
        events.append(_make_event(user_id, dtype, late_stage, new_choice, ts))
    return events

def _build_chronic_indecisive(user_id: str, session_start: datetime, config: Dict[str, Any] = CONFIG) -> List[dict]:
    events = []
    ts = session_start
    for dtype in config["DECISION_TYPES"]:
        n_changes = random.randint(2, 4)
        stage_iter = iter(config["STAGE_ORDER"])
        current_stage = next(stage_iter, "Browsing")
        choice = random.choice(config["CHOICE_SPACE"][dtype])
        ts += timedelta(seconds=random.randint(20, 60))
        events.append(_make_event(user_id, dtype, current_stage, choice, ts))

        for _ in range(n_changes):
            ts += timedelta(seconds=random.randint(60, 300))
            if random.random() < 0.5:
                current_stage = next(stage_iter, config["STAGE_ORDER"][-1])
            new_choice = _other_choice(choice, dtype, config)
            events.append(_make_event(user_id, dtype, current_stage, new_choice, ts))
            choice = new_choice
    return events

BUILDERS = {
    "decisive": _build_decisive,
    "single_corrector": _build_single_corrector,
    "flip_flopper": _build_flip_flopper,
    "late_reverser": _build_late_reverser,
    "chronic_indecisive": _build_chronic_indecisive,
}

def generate_synthetic_dataset(
    num_users: int = 200,
    seed: int = 42,
    noise_rate: float = 0.0,
    config: Dict[str, Any] = CONFIG,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generates synthetic events and ground truth personas faithful to data/generate_data.py."""
    random.seed(seed)
    np.random.seed(seed)
    
    personas_list = []
    for p in ["decisive", "single_corrector", "flip_flopper", "late_reverser", "chronic_indecisive"]:
        personas_list.extend([p] * (num_users // 5))
    random.shuffle(personas_list)

    all_events = []
    truth_rows = []
    base_date = datetime(2026, 6, 1, 8, 0, 0)

    for i, persona in enumerate(personas_list):
        user_id = f"U{i + 1:04d}"
        offset_hours = random.uniform(0, 30 * 24)
        session_start = base_date + timedelta(hours=offset_hours)
        builder = BUILDERS[persona]
        user_events = builder(user_id, session_start, config)

        if noise_rate > 0.0 and random.random() < noise_rate:
            d = random.choice(config["DECISION_TYPES"])
            c = random.choice(config["CHOICE_SPACE"][d])
            s = random.choice(config["STAGE_ORDER"])
            ts = session_start + timedelta(seconds=random.randint(10, 500))
            user_events.append(_make_event(user_id, d, s, c, ts))

        all_events.extend(user_events)
        truth_rows.append({"user_id": user_id, "persona": persona})

    events_df = pd.DataFrame(all_events)
    events_df["timestamp"] = pd.to_datetime(events_df["timestamp"])
    events_df = events_df.sort_values("timestamp").reset_index(drop=True)
    gt_df = pd.DataFrame(truth_rows)

    os.makedirs("data", exist_ok=True)
    events_df.to_csv("data/sample_events.csv", index=False)
    gt_df.to_csv("data/ground_truth.csv", index=False)
    return events_df, gt_df

# Generate Primary Dataset (Seed 42)
raw_events_df, ground_truth_df = generate_synthetic_dataset(num_users=200, seed=CONFIG["SEED"])

print(f"Generated {len(raw_events_df):,} events across {ground_truth_df['user_id'].nunique()} users.")
print(f"Saved: data/sample_events.csv and data/ground_truth.csv")
print("\nPersona Distribution in Ground Truth:")
display(ground_truth_df["persona"].value_counts())
print("\nSample Raw Events (Head):")
display(raw_events_df.head())

Generated 1,239 events across 200 users.
Saved: data/sample_events.csv and data/ground_truth.csv

Persona Distribution in Ground Truth:


persona
single_corrector      40
chronic_indecisive    40
flip_flopper          40
late_reverser         40
decisive              40
Name: count, dtype: int64


Sample Raw Events (Head):


,user_id,decision_id,timestamp,stage,choice
0,U0154,U0154_plan,2026-06-01 09:04:25,Selection,enterprise
1,U0154,U0154_add_on,2026-06-01 09:05:08,Selection,none
2,U0154,U0154_payment_method,2026-06-01 09:06:04,Review,debit_card
3,U0154,U0154_payment_method,2026-06-01 09:06:40,Review,credit_card
4,U0099,U0099_payment_method,2026-06-01 11:44:46,Browsing,crypto


## Section 3: Loader & Schema Validation
The loader guarantees data integrity before analytical processing:
1. Enforces required columns: `['user_id', 'decision_id', 'timestamp', 'stage', 'choice']`.
2. Validates stages against the canonical sequence `STAGE_ORDER`.
3. Parses timestamps into timezone-naive `datetime64[ns]` objects.
4. Sorts events chronologically per `(user_id, timestamp)` to ensure valid causal transitions.
5. Catches and reports specific schema violations gracefully.

In [4]:
def load_and_validate(source: Any, config: Dict[str, Any] = CONFIG) -> pd.DataFrame:
    """
    Ingests and validates raw event logs from a CSV file path, buffer, or existing DataFrame.
    
    Enforces:
      - Presence of all required columns
      - Disallowing missing/null values
      - Strict membership of 'stage' in config['STAGE_ORDER']
      - ISO-8601 timestamp parsing and chronological sorting
    """
    required_cols = config["CSV_COLUMNS"]
    ts_col = config["TIMESTAMP_COL"]
    stages = config["STAGE_ORDER"]
    
    if isinstance(source, pd.DataFrame):
        df = source.copy()
    elif isinstance(source, (str, pathlib.Path)):
        if not os.path.exists(source):
            raise FileNotFoundError(f"File not found: {source}")
        df = pd.read_csv(source)
    elif isinstance(source, (io.StringIO, io.BytesIO)):
        source.seek(0)
        df = pd.read_csv(source)
    else:
        raise ValueError(f"Unsupported source type: {type(source)}")
    
    # 1. Column presence check
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Available: {list(df.columns)}")
    
    df = df[required_cols].copy()
    
    # 2. Null check
    nulls = df.isnull().sum()
    bad_cols = nulls[nulls > 0].to_dict()
    if bad_cols:
        raise ValueError(f"Null values detected in columns: {bad_cols}")
    
    # 3. Stage membership check
    invalid_stages = set(df["stage"].unique()) - set(stages)
    if invalid_stages:
        raise ValueError(f"Unrecognized stage(s) {invalid_stages}. Expected one of: {stages}")
    
    # 4. Timestamp parsing
    try:
        df[ts_col] = pd.to_datetime(df[ts_col])
    except Exception as e:
        raise ValueError(f"Failed to parse timestamps in '{ts_col}': {e}")
    
    # 5. Chronological sort per user
    df = df.sort_values(by=["user_id", ts_col], ascending=[True, True]).reset_index(drop=True)
    return df

# Validate on primary dataset
clean_events_df = load_and_validate("data/sample_events.csv", CONFIG)
print(f"✅ Successfully validated {len(clean_events_df):,} events. All timestamps parsed.")

# Demonstration of deliberate bad input catching
print("\n--- Negative Test: Intentionally Malformed CSV ---")
bad_csv = """user_id,decision_id,timestamp,stage,choice
U001,plan,2026-09-01 10:00:00,InvalidStage,pro
"""
try:
    load_and_validate(io.StringIO(bad_csv), CONFIG)
except ValueError as err:
    print(f"✅ Caught expected validation error: {err}")

✅ Successfully validated 1,239 events. All timestamps parsed.

--- Negative Test: Intentionally Malformed CSV ---
✅ Caught expected validation error: Unrecognized stage(s) {'InvalidStage'}. Expected one of: ['Browsing', 'Selection', 'Review', 'Confirmation', 'Post-commit']


## Section 4: Reversal Detection Engine
For every `(user_id, decision_id)` pair, the detector evaluates the chronological choice stream:
- Identifies choice transitions where $c_t \neq c_{t-1}$.
- Calculates the hesitation gap $\Delta t = t_{\text{curr}} - t_{\text{prev}}$ in seconds.
- Detects **flip-backs**: $c_t \in \{c_1, c_2, \dots, c_{t-1}\}$, meaning the user returned to an option they had previously discarded.
- Returns a structured DataFrame with one row per detected reversal.

In [5]:
def detect_reversals(df: pd.DataFrame, config: Dict[str, Any] = CONFIG) -> pd.DataFrame:
    """
    Detects choice reversals and flip-backs from validated event log.
    
    Returns DataFrame with columns:
      - user_id, decision_id, from_choice, to_choice, stage, timestamp, seconds_since_previous, is_flip_back
    """
    reversals = []
    ts_col = config["TIMESTAMP_COL"]
    
    # Group by user and decision in chronological order
    for (uid, dec_id), group in df.groupby(["user_id", "decision_id"], sort=False):
        choices = group["choice"].tolist()
        timestamps = group[ts_col].tolist()
        stages = group["stage"].tolist()
        
        seen_choices = []
        for i in range(len(choices)):
            curr_c = choices[i]
            curr_t = timestamps[i]
            curr_s = stages[i]
            
            if i > 0:
                prev_c = choices[i - 1]
                prev_t = timestamps[i - 1]
                
                # A reversal occurs if current choice differs from immediately previous choice
                if curr_c != prev_c:
                    delta_sec = max(0.0, (curr_t - prev_t).total_seconds())
                    is_flip = curr_c in seen_choices
                    
                    reversals.append({
                        "user_id": uid,
                        "decision_id": dec_id,
                        "from_choice": prev_c,
                        "to_choice": curr_c,
                        "stage": curr_s,
                        "timestamp": curr_t,
                        "seconds_since_previous": delta_sec,
                        "is_flip_back": is_flip,
                    })
            
            seen_choices.append(curr_c)
            
    rev_df = pd.DataFrame(reversals)
    if rev_df.empty:
        return pd.DataFrame(columns=[
            "user_id", "decision_id", "from_choice", "to_choice",
            "stage", "timestamp", "seconds_since_previous", "is_flip_back"
        ])
    return rev_df

reversals_df = detect_reversals(clean_events_df, CONFIG)
print(f"✅ Reversal detection complete: {len(reversals_df):,} reversals found across {reversals_df['user_id'].nunique()} users.")
print(f"   Flip-backs detected: {reversals_df['is_flip_back'].sum():,}")
print("\nSample Detected Reversals:")
display(reversals_df.head(6))

✅ Reversal detection complete: 639 reversals found across 160 users.
   Flip-backs detected: 223

Sample Detected Reversals:


,user_id,decision_id,from_choice,to_choice,stage,timestamp,seconds_since_previous,is_flip_back
0,U0001,U0001_payment_method,paypal,debit_card,Review,2026-06-30 06:23:41,38.0,False
1,U0002,U0002_plan,basic,pro,Browsing,2026-06-04 23:09:08,65.0,False
2,U0002,U0002_plan,pro,basic,Selection,2026-06-04 23:12:38,210.0,True
3,U0002,U0002_add_on,none,priority_support,Browsing,2026-06-04 23:14:29,77.0,False
4,U0002,U0002_add_on,priority_support,analytics,Browsing,2026-06-04 23:15:47,78.0,False
5,U0002,U0002_add_on,analytics,all_inclusive,Selection,2026-06-04 23:19:38,231.0,False


## Section 5: Per-User Metric Aggregation
We aggregate per-user behavioral KPIs across all decisions:
- `total_reversals`: Count of all reversals made by the user.
- `flip_back_count`: Count of circular return transitions.
- `reversal_rate`: `total_reversals / num_distinct_decisions_made`.
- `decisions_with_reversals`: Number of distinct decisions that experienced $\ge 1$ reversal.
- `max_share_on_one_decision`: Largest single-decision share of the user's total reversals (concentration measure).
- `avg_hesitation_seconds`: Mean hesitation delta (NaN for users with 0 reversals).
- `most_common_reversal_stage`: Funnel stage with highest reversal frequency.
- `stage_distribution`: Dictionary of reversal counts per stage.

In [6]:
def compute_metrics(
    events_df: pd.DataFrame,
    reversals_df: pd.DataFrame,
    config: Dict[str, Any] = CONFIG,
) -> pd.DataFrame:
    """
    Computes summary behavioral metrics for every user found in events_df.
    Guarantees users with zero reversals are safely preserved without division-by-zero errors.
    """
    stages = config["STAGE_ORDER"]
    all_users = sorted(events_df["user_id"].unique())
    user_rows = []
    
    for uid in all_users:
        u_events = events_df[events_df["user_id"] == uid]
        u_revs = reversals_df[reversals_df["user_id"] == uid] if not reversals_df.empty else pd.DataFrame()
        
        n_decisions = u_events["decision_id"].nunique()
        n_reversals = len(u_revs)
        
        # Per-stage counts
        stage_counts = {s: 0 for s in stages}
        if not u_revs.empty:
            for s, cnt in u_revs["stage"].value_counts().items():
                if s in stage_counts:
                    stage_counts[s] = int(cnt)
        
        # Per-decision reversal counts
        dec_counts: Dict[str, int] = {}
        if not u_revs.empty:
            dec_counts = u_revs["decision_id"].value_counts().to_dict()
            
        decisions_with_revs = len(dec_counts)
        max_share = (max(dec_counts.values()) / n_reversals) if n_reversals > 0 else 0.0
        
        # Safe rate computation
        rev_rate = (n_reversals / n_decisions) if n_decisions > 0 else 0.0
        flip_backs = int(u_revs["is_flip_back"].sum()) if not u_revs.empty else 0
        avg_hesitation = float(u_revs["seconds_since_previous"].mean()) if n_reversals > 0 else float("nan")
        
        # Modal stage
        most_common_stage = ""
        if n_reversals > 0:
            stage_series = u_revs["stage"].value_counts()
            top_cnt = stage_series.iloc[0]
            # preserve canonical stage order in case of ties
            for s in stages:
                if stage_counts.get(s, 0) == top_cnt:
                    most_common_stage = s
                    break
        
        user_rows.append({
            "user_id": uid,
            "total_reversals": n_reversals,
            "reversals_per_decision": dec_counts,
            "flip_back_count": flip_backs,
            "reversal_rate": round(rev_rate, 4),
            "decisions_with_reversals": decisions_with_revs,
            "max_share_on_one_decision": round(max_share, 4),
            "avg_hesitation_seconds": round(avg_hesitation, 2) if not math.isnan(avg_hesitation) else np.nan,
            "most_common_reversal_stage": most_common_stage,
            "stage_distribution": stage_counts,
        })
        
    return pd.DataFrame(user_rows)

metrics_df = compute_metrics(clean_events_df, reversals_df, CONFIG)
print(f"✅ Computed metrics for {len(metrics_df)} users.")
display(metrics_df.head(6))

✅ Computed metrics for 200 users.


,user_id,total_reversals,reversals_per_decision,flip_back_count,reversal_rate,decisions_with_reversals,max_share_on_one_decision,avg_hesitation_seconds,most_common_reversal_stage,stage_distribution
0,U0001,1,{'U0001_payment_method': 1},0,0.3333,1,1.0000,38.00,Review,"{'Browsing': 0, 'Selection': 0, 'Review': 1, '..."
1,U0002,10,"{'U0002_add_on': 4, 'U0002_payment_method': 4,...",4,3.3333,3,0.4000,147.10,Selection,"{'Browsing': 3, 'Selection': 4, 'Review': 1, '..."
2,U0003,7,"{'U0003_payment_method': 5, 'U0003_plan': 1, '...",3,2.3333,3,0.7143,104.71,Review,"{'Browsing': 0, 'Selection': 0, 'Review': 3, '..."
3,U0004,9,"{'U0004_add_on': 4, 'U0004_plan': 3, 'U0004_pa...",2,3.0000,3,0.4444,149.11,Browsing,"{'Browsing': 6, 'Selection': 3, 'Review': 0, '..."
4,U0005,5,{'U0005_add_on': 5},2,1.6667,1,1.0000,95.80,Post-commit,"{'Browsing': 0, 'Selection': 0, 'Review': 1, '..."
5,U0006,1,{'U0006_payment_method': 1},0,0.3333,1,1.0000,566.00,Post-commit,"{'Browsing': 0, 'Selection': 0, 'Review': 0, '..."


## Section 6: Rule-Based Pattern Classifier
The classifier assigns one of 6 mutually exclusive behavioral pattern labels using strict priority evaluation:

1. **`decisive`**: 0 reversals (`total_reversals == 0`).
2. **`single_corrector`**: Exactly 1 reversal overall, `flip_back_count == 0`, hesitation $\le 60\text{s}$.
3. **`chronic_indecisive`**: Broad indecision: `reversal_rate >= 0.75` across $\ge 2$ decisions, with `max_share_on_one_decision < 0.60`.
4. **`flip_flopper`**: Concentrated oscillation: $(\ge 3\text{ reversals on 1 decision OR } \ge 2\text{ flip-backs})$ AND `max_share_on_one_decision >= 0.60`.
5. **`late_reverser`**: $\ge 60\%$ of reversals occurring at `Confirmation` or `Post-commit`.
6. **`mixed`**: Any remaining behavioral patterns.

In [7]:
# Pattern Constants
PATTERN_DECISIVE           = "decisive"
PATTERN_SINGLE_CORRECTOR   = "single_corrector"
PATTERN_CHRONIC_INDECISIVE = "chronic_indecisive"
PATTERN_FLIP_FLOPPER       = "flip_flopper"
PATTERN_LATE_REVERSER      = "late_reverser"
PATTERN_MIXED              = "mixed"

def classify_user_pattern(
    row: pd.Series,
    user_revs: pd.DataFrame,
    config: Dict[str, Any] = CONFIG,
) -> str:
    """Evaluates a single user against priority rules."""
    n_revs = row["total_reversals"]
    
    # Priority 1: Decisive
    if n_revs == 0:
        return PATTERN_DECISIVE
        
    # Priority 2: Single Corrector
    if n_revs == 1 and row["flip_back_count"] == 0:
        if not user_revs.empty:
            gap = user_revs["seconds_since_previous"].iloc[0]
            if gap <= config["CORRECTION_MAX_SECONDS"]:
                return PATTERN_SINGLE_CORRECTOR
                
    # Priority 3: Chronic Indecisive (Broad churn)
    if (
        row["reversal_rate"] >= config["CHRONIC_INDECISIVE_MIN_REVERSAL_RATE"]
        and row["decisions_with_reversals"] >= config["CHRONIC_INDECISIVE_MIN_DECISIONS"]
        and row["max_share_on_one_decision"] < config["CHRONIC_INDECISIVE_MAX_SHARE"]
    ):
        return PATTERN_CHRONIC_INDECISIVE
        
    # Priority 4: Flip-Flopper (Concentrated churn)
    revs_per_dec = row["reversals_per_decision"]
    max_on_one = max(revs_per_dec.values()) if revs_per_dec else 0
    has_oscillation = (
        max_on_one >= config["FLIP_FLOPPER_MIN_REVERSALS_ON_ONE_DECISION"]
        or row["flip_back_count"] >= config["FLIP_FLOPPER_MIN_FLIP_BACKS"]
    )
    if has_oscillation and row["max_share_on_one_decision"] >= config["FLIP_FLOPPER_MIN_SHARE"]:
        return PATTERN_FLIP_FLOPPER
        
    # Priority 5: Late Reverser
    if not user_revs.empty:
        late_count = user_revs["stage"].isin(config["LATE_STAGES"]).sum()
        late_share = late_count / n_revs
        if late_share >= config["LATE_REVERSER_MIN_LATE_SHARE"]:
            return PATTERN_LATE_REVERSER
            
    # Priority 6: Mixed
    return PATTERN_MIXED

def classify_all_users(
    metrics_df: pd.DataFrame,
    reversals_df: pd.DataFrame,
    config: Dict[str, Any] = CONFIG,
) -> pd.DataFrame:
    """Assigns pattern labels to all users in metrics_df."""
    classified = metrics_df.copy()
    patterns = []
    
    for _, row in classified.iterrows():
        uid = row["user_id"]
        u_revs = reversals_df[reversals_df["user_id"] == uid] if not reversals_df.empty else pd.DataFrame()
        p = classify_user_pattern(row, u_revs, config)
        patterns.append(p)
        
    classified["pattern"] = patterns
    return classified

classified_df = classify_all_users(metrics_df, reversals_df, CONFIG)
print("✅ Classification complete. Pattern Distribution:")
display(classified_df["pattern"].value_counts())

✅ Classification complete. Pattern Distribution:


pattern
single_corrector      40
chronic_indecisive    40
flip_flopper          40
late_reverser         40
decisive              40
Name: count, dtype: int64

## Section 7: Human-Readable Explanations
Generates transparent, deterministic, one-sentence plain English evidence for every user.
Implemented with zero external LLMs or black-box APIs, ensuring 100% auditability and zero cost per inference.

In [8]:
def explain_user(
    uid: str,
    row: pd.Series,
    u_revs: pd.DataFrame,
    pattern: str,
    config: Dict[str, Any] = CONFIG,
) -> str:
    """Formats a plain-English evidence sentence for a user."""
    n_revs = row["total_reversals"]
    
    if pattern == PATTERN_DECISIVE:
        return f"{uid} made no changes to any decision."
        
    if pattern == PATTERN_SINGLE_CORRECTOR:
        if not u_revs.empty:
            r = u_revs.iloc[0]
            sec = int(round(r["seconds_since_previous"]))
            return (
                f"{uid} made a quick correction to '{r['decision_id']}' within {sec} s "
                f"(changed from '{r['from_choice']}' to '{r['to_choice']}') and did not look back."
            )
        return f"{uid} made a quick single correction and did not look back."
        
    if pattern == PATTERN_FLIP_FLOPPER:
        dec_counts = row["reversals_per_decision"]
        focal_dec = max(dec_counts, key=dec_counts.get) if dec_counts else "choice"
        n_focal = dec_counts.get(focal_dec, n_revs)
        n_flips = row["flip_back_count"]
        top_stage = row["most_common_reversal_stage"] or "Selection"
        return (
            f"{uid} changed '{focal_dec}' {n_focal} times, {n_flips} of them back to a previous choice, "
            f"mostly at the {top_stage} stage."
        )
        
    if pattern == PATTERN_LATE_REVERSER:
        late_stages = config["LATE_STAGES"]
        late_revs = u_revs[u_revs["stage"].isin(late_stages)] if not u_revs.empty else pd.DataFrame()
        n_late = len(late_revs)
        n_dec = late_revs["decision_id"].nunique() if not late_revs.empty else 1
        top_stage = row["most_common_reversal_stage"] or "Confirmation"
        return (
            f"{uid} reversed {n_dec} decision(s) late in the funnel: {n_late} at {top_stage} "
            f"(out of {n_revs} reversal(s) total)."
        )
        
    if pattern == PATTERN_CHRONIC_INDECISIVE:
        n_dec = row["decisions_with_reversals"]
        rate = row["reversal_rate"]
        return (
            f"{uid} changed decisions broadly across {n_dec} area(s) with an overall reversal rate "
            f"of {rate:.2f} changes per decision."
        )
        
    # Mixed / fallback
    return (
        f"{uid} made {n_revs} reversal(s) across {row['decisions_with_reversals']} decision(s) "
        f"with {row['flip_back_count']} flip-back(s)."
    )

def explain_all_users(
    classified_df: pd.DataFrame,
    reversals_df: pd.DataFrame,
    config: Dict[str, Any] = CONFIG,
) -> pd.DataFrame:
    """Attaches explanation column to classified DataFrame."""
    res = classified_df.copy()
    sentences = []
    for _, row in res.iterrows():
        uid = row["user_id"]
        u_revs = reversals_df[reversals_df["user_id"] == uid] if not reversals_df.empty else pd.DataFrame()
        s = explain_user(uid, row, u_revs, row["pattern"], config)
        sentences.append(s)
    res["explanation"] = sentences
    return res

explained_df = explain_all_users(classified_df, reversals_df, CONFIG)

print("✅ Generated natural evidence sentences. Sample outputs per persona:")
for p in [PATTERN_DECISIVE, PATTERN_SINGLE_CORRECTOR, PATTERN_FLIP_FLOPPER, PATTERN_LATE_REVERSER, PATTERN_CHRONIC_INDECISIVE]:
    sample_row = explained_df[explained_df["pattern"] == p].iloc[0]
    print(f"\n[{p.upper()}] - {sample_row['user_id']}:")
    print(f"  \"{sample_row['explanation']}\"")

✅ Generated natural evidence sentences. Sample outputs per persona:

[DECISIVE] - U0007:
  "U0007 made no changes to any decision."

[SINGLE_CORRECTOR] - U0001:
  "U0001 made a quick correction to 'U0001_payment_method' within 38 s (changed from 'paypal' to 'debit_card') and did not look back."

[FLIP_FLOPPER] - U0003:
  "U0003 changed 'U0003_payment_method' 5 times, 3 of them back to a previous choice, mostly at the Review stage."

[LATE_REVERSER] - U0006:
  "U0006 reversed 1 decision(s) late in the funnel: 1 at Post-commit (out of 1 reversal(s) total)."

[CHRONIC_INDECISIVE] - U0002:
  "U0002 changed decisions broadly across 3 area(s) with an overall reversal rate of 3.33 changes per decision."


## Section 8: Plotly Visualisations
We provide three interactive Plotly visualization functions:
1. `timeline_chart`: Chronological choice paths per decision, with red reversal markers and annotations.
2. `stage_chart`: Global horizontal bar chart of reversals distributed across the funnel.
3. `flow_chart`: Interactive Sankey diagram showing choice-to-choice transitions and cyclic loops.

In [9]:
def timeline_chart(events_df: pd.DataFrame, reversals_df: pd.DataFrame, user_id: str) -> go.Figure:
    """Generates timeline of choices over time with reversal points marked."""
    u_events = events_df[events_df["user_id"] == user_id].sort_values("timestamp")
    u_revs = reversals_df[reversals_df["user_id"] == user_id] if not reversals_df.empty else pd.DataFrame()
    
    fig = go.Figure()
    decisions = u_events["decision_id"].unique()
    colors = px.colors.qualitative.Plotly
    
    for i, dec in enumerate(decisions):
        sub = u_events[u_events["decision_id"] == dec]
        fig.add_trace(go.Scatter(
            x=sub["timestamp"],
            y=sub["choice"],
            mode="lines+markers",
            name=str(dec),
            line=dict(width=2, color=colors[i % len(colors)]),
            marker=dict(size=8),
        ))
        
    if not u_revs.empty:
        fig.add_trace(go.Scatter(
            x=u_revs["timestamp"],
            y=u_revs["to_choice"],
            mode="markers+text",
            name="Reversal Event",
            marker=dict(symbol="circle-open", size=14, color="#ef4444", line=dict(width=3)),
            text=[f"↩ {r.from_choice}→{r.to_choice}" for _, r in u_revs.iterrows()],
            textposition="top center",
        ))
        
    fig.update_layout(
        title=f"Decision Trajectory Timeline — User {user_id}",
        xaxis_title="Event Timestamp",
        yaxis_title="Selected Choice",
        template="plotly_dark",
        height=380,
        margin=dict(l=40, r=40, t=50, b=40),
    )
    return fig

def stage_chart(reversals_df: pd.DataFrame, config: Dict[str, Any] = CONFIG) -> go.Figure:
    """Bar chart of reversal counts across funnel stages."""
    stages = config["STAGE_ORDER"]
    counts = {s: 0 for s in stages}
    if not reversals_df.empty:
        for s, c in reversals_df["stage"].value_counts().items():
            if s in counts:
                counts[s] = int(c)
                
    fig = go.Figure(go.Bar(
        x=stages,
        y=[counts[s] for s in stages],
        marker=dict(color=["#6366f1", "#8b5cf6", "#ec4899", "#f59e0b", "#ef4444"]),
        text=[counts[s] for s in stages],
        textposition="auto",
    ))
    fig.update_layout(
        title="Reversal Distribution Across Funnel Stages",
        xaxis_title="Checkout Funnel Stage",
        yaxis_title="Reversals Count",
        template="plotly_dark",
        height=340,
        margin=dict(l=40, r=40, t=50, b=40),
    )
    return fig

def flow_chart(events_df: pd.DataFrame, user_id: str, decision_id: str) -> go.Figure:
    """Sankey diagram showing transitions and cycles between choices for a decision."""
    sub = events_df[(events_df["user_id"] == user_id) & (events_df["decision_id"] == decision_id)].sort_values("timestamp")
    choices = sub["choice"].tolist()
    
    if len(choices) < 2:
        fig = go.Figure()
        fig.update_layout(title=f"No transitions recorded for {user_id} on '{decision_id}'", template="plotly_dark", height=300)
        return fig
        
    all_nodes = sorted(list(set(choices)))
    node_idx = {c: i for i, c in enumerate(all_nodes)}
    
    transitions: Dict[Tuple[int, int], int] = {}
    for i in range(len(choices) - 1):
        src = node_idx[choices[i]]
        tgt = node_idx[choices[i + 1]]
        pair = (src, tgt)
        transitions[pair] = transitions.get(pair, 0) + 1
        
    sources = [p[0] for p in transitions]
    targets = [p[1] for p in transitions]
    values = [transitions[p] for p in transitions]
    
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=all_nodes,
            color="#6366f1",
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color="rgba(168, 85, 247, 0.4)",
        )
    )])
    fig.update_layout(
        title=f"Choice Flow Transitions — User {user_id} on '{decision_id}'",
        font_size=12,
        template="plotly_dark",
        height=320,
        margin=dict(l=40, r=40, t=50, b=40),
    )
    return fig

# Render global stage chart and sample user visualization
print("Rendering Global Stage Chart:")
stage_chart(reversals_df, CONFIG).show()

# Render sample flip-flopper user (e.g. U0081)
sample_flip_user = explained_df[explained_df["pattern"] == PATTERN_FLIP_FLOPPER].iloc[0]["user_id"]
print(f"\nRendering Timeline Chart for Flip-Flopper {sample_flip_user}:")
timeline_chart(clean_events_df, reversals_df, sample_flip_user).show()

Rendering Global Stage Chart:



Rendering Timeline Chart for Flip-Flopper U0003:


## Section 9: Interactive Explorer (ipywidgets)
Select any user from the interactive dropdown below to view their real-time metrics card, pattern classification, plain English evidence sentence, timeline chart, and choice flow diagram.

*Note: In headless mode or "Run All", the widget initializes non-blockingly.*

In [10]:
user_list = explained_df["user_id"].tolist()
dropdown = widgets.Dropdown(
    options=user_list,
    value=user_list[41] if len(user_list) > 41 else user_list[0],  # default to a user with reversals
    description="Select User:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="320px"),
)

out = widgets.Output()

def on_user_change(change):
    uid = change["new"]
    with out:
        clear_output(wait=True)
        row = explained_df[explained_df["user_id"] == uid].iloc[0]
        
        # HTML Metric Summary Card
        badge_color = {
            PATTERN_DECISIVE: "#10b981",
            PATTERN_SINGLE_CORRECTOR: "#3b82f6",
            PATTERN_CHRONIC_INDECISIVE: "#a855f7",
            PATTERN_FLIP_FLOPPER: "#ef4444",
            PATTERN_LATE_REVERSER: "#f59e0b",
            PATTERN_MIXED: "#94a3b8",
        }.get(row["pattern"], "#6366f1")
        
        html_card = f"""
        <div style="background:#1e293b; padding:18px; border-radius:12px; border:1px solid #334155; margin-bottom:12px;">
            <div style="display:flex; justify-content:space-between; align-items:center;">
                <h3 style="color:#f8fafc; margin:0;">User Profile: {uid}</h3>
                <span style="background:{badge_color}; color:#fff; padding:4px 12px; border-radius:16px; font-weight:bold; font-size:0.85rem;">
                    {row['pattern'].upper()}
                </span>
            </div>
            <p style="color:#cbd5e1; font-size:1.05rem; margin:10px 0 14px 0;"><strong>Evidence:</strong> <em>{row['explanation']}</em></p>
            <div style="display:grid; grid-template-columns: repeat(4, 1fr); gap:12px;">
                <div style="background:#0f172a; padding:10px; border-radius:8px; text-align:center;">
                    <div style="color:#94a3b8; font-size:0.75rem;">Total Reversals</div>
                    <div style="color:#f8fafc; font-size:1.3rem; font-weight:bold;">{row['total_reversals']}</div>
                </div>
                <div style="background:#0f172a; padding:10px; border-radius:8px; text-align:center;">
                    <div style="color:#94a3b8; font-size:0.75rem;">Reversal Rate</div>
                    <div style="color:#f8fafc; font-size:1.3rem; font-weight:bold;">{row['reversal_rate']:.2f}</div>
                </div>
                <div style="background:#0f172a; padding:10px; border-radius:8px; text-align:center;">
                    <div style="color:#94a3b8; font-size:0.75rem;">Flip-Backs</div>
                    <div style="color:#f8fafc; font-size:1.3rem; font-weight:bold;">{row['flip_back_count']}</div>
                </div>
                <div style="background:#0f172a; padding:10px; border-radius:8px; text-align:center;">
                    <div style="color:#94a3b8; font-size:0.75rem;">Avg Hesitation</div>
                    <div style="color:#f8fafc; font-size:1.3rem; font-weight:bold;">{row['avg_hesitation_seconds']} s</div>
                </div>
            </div>
        </div>
        """
        display(HTML(html_card))
        
        # Display Timeline
        fig_t = timeline_chart(clean_events_df, reversals_df, uid)
        fig_t.show()
        
        # Display Flow Chart for user's most active decision
        u_events = clean_events_df[clean_events_df["user_id"] == uid]
        if not u_events.empty:
            focal_dec = u_events["decision_id"].value_counts().index[0]
            fig_f = flow_chart(clean_events_df, uid, focal_dec)
            fig_f.show()

dropdown.observe(on_user_change, names="value")
display(widgets.VBox([dropdown, out]))

# Trigger initial rendering for non-interactive / headless execution
on_user_change({"new": dropdown.value})

## Section 10: Inline Verification & Assert-Based Unit Tests
We execute automated mathematical unit tests covering core behaviors and edge cases:
1. `test_no_change`: User makes 1 choice per decision $\to$ 0 reversals, `decisive`.
2. `test_single_correction`: Exactly 1 reversal within 60s without flip-backs $\to$ `single_corrector`.
3. `test_flip_back`: $A \to B \to A$ oscillation $\to$ `is_flip_back == True`.
4. `test_linear_progression`: $A \to B \to C$ choice sequence $\to$ 2 reversals, but 0 flip-backs.
5. `test_late_reverser`: Reversals occurring at `Confirmation` $\to$ `late_reverser`.
6. `test_chronic_indecision`: Broad reversals across $\ge 2$ decisions $\to$ `chronic_indecisive`.
7. `test_zero_reversals_metrics`: Division-by-zero safeguard and NaN hesitation handling.
8. `test_empty_input`: Graceful empty DataFrame handling.

In [11]:
def run_unit_tests():
    print("Running inline assert tests...")
    
    # Test 1: No change (Decisive)
    df_no_change = pd.DataFrame([
        {"user_id": "T01", "decision_id": "plan", "timestamp": "2026-09-01 10:00:00", "stage": "Selection", "choice": "pro"},
        {"user_id": "T01", "decision_id": "add_on", "timestamp": "2026-09-01 10:01:00", "stage": "Selection", "choice": "analytics"},
    ])
    revs_1 = detect_reversals(load_and_validate(df_no_change, CONFIG), CONFIG)
    assert len(revs_1) == 0, f"Expected 0 reversals, got {len(revs_1)}"
    m_1 = compute_metrics(load_and_validate(df_no_change, CONFIG), revs_1, CONFIG)
    c_1 = classify_all_users(m_1, revs_1, CONFIG)
    assert c_1["pattern"].iloc[0] == PATTERN_DECISIVE
    print("  [1/8] test_no_change PASSED")
    
    # Test 2: Single quick correction
    df_single = pd.DataFrame([
        {"user_id": "T02", "decision_id": "plan", "timestamp": "2026-09-01 10:00:00", "stage": "Selection", "choice": "basic"},
        {"user_id": "T02", "decision_id": "plan", "timestamp": "2026-09-01 10:00:25", "stage": "Selection", "choice": "pro"},
    ])
    revs_2 = detect_reversals(load_and_validate(df_single, CONFIG), CONFIG)
    assert len(revs_2) == 1
    assert revs_2["is_flip_back"].iloc[0] == False
    assert revs_2["seconds_since_previous"].iloc[0] == 25.0
    m_2 = compute_metrics(load_and_validate(df_single, CONFIG), revs_2, CONFIG)
    c_2 = classify_all_users(m_2, revs_2, CONFIG)
    assert c_2["pattern"].iloc[0] == PATTERN_SINGLE_CORRECTOR
    print("  [2/8] test_single_correction PASSED")
    
    # Test 3: A -> B -> A flip-back
    df_flip = pd.DataFrame([
        {"user_id": "T03", "decision_id": "plan", "timestamp": "2026-09-01 10:00:00", "stage": "Selection", "choice": "basic"},
        {"user_id": "T03", "decision_id": "plan", "timestamp": "2026-09-01 10:01:00", "stage": "Selection", "choice": "pro"},
        {"user_id": "T03", "decision_id": "plan", "timestamp": "2026-09-01 10:02:00", "stage": "Selection", "choice": "basic"},
    ])
    revs_3 = detect_reversals(load_and_validate(df_flip, CONFIG), CONFIG)
    assert len(revs_3) == 2
    assert revs_3["is_flip_back"].iloc[0] == False
    assert revs_3["is_flip_back"].iloc[1] == True  # returned to 'basic'
    print("  [3/8] test_flip_back PASSED")
    
    # Test 4: A -> B -> C linear progression (no flip-back)
    df_prog = pd.DataFrame([
        {"user_id": "T04", "decision_id": "plan", "timestamp": "2026-09-01 10:00:00", "stage": "Selection", "choice": "basic"},
        {"user_id": "T04", "decision_id": "plan", "timestamp": "2026-09-01 10:01:00", "stage": "Selection", "choice": "pro"},
        {"user_id": "T04", "decision_id": "plan", "timestamp": "2026-09-01 10:02:00", "stage": "Selection", "choice": "enterprise"},
    ])
    revs_4 = detect_reversals(load_and_validate(df_prog, CONFIG), CONFIG)
    assert len(revs_4) == 2
    assert revs_4["is_flip_back"].sum() == 0  # no choice repeated
    print("  [4/8] test_linear_progression PASSED")
    
    # Test 5: Late reverser
    df_late = pd.DataFrame([
        {"user_id": "T05", "decision_id": "plan", "timestamp": "2026-09-01 10:00:00", "stage": "Browsing", "choice": "basic"},
        {"user_id": "T05", "decision_id": "plan", "timestamp": "2026-09-01 10:05:00", "stage": "Confirmation", "choice": "pro"},
    ])
    revs_5 = detect_reversals(load_and_validate(df_late, CONFIG), CONFIG)
    m_5 = compute_metrics(load_and_validate(df_late, CONFIG), revs_5, CONFIG)
    # Gap is 300s (>60s), so single_corrector is bypassed, falls into late_reverser
    c_5 = classify_all_users(m_5, revs_5, CONFIG)
    assert c_5["pattern"].iloc[0] == PATTERN_LATE_REVERSER
    print("  [5/8] test_late_reverser PASSED")
    
    # Test 6: Chronic Indecision (Broad churn across decisions)
    df_chronic = pd.DataFrame([
        {"user_id": "T06", "decision_id": "plan", "timestamp": "2026-09-01 10:00:00", "stage": "Browsing", "choice": "basic"},
        {"user_id": "T06", "decision_id": "plan", "timestamp": "2026-09-01 10:02:00", "stage": "Selection", "choice": "pro"},
        {"user_id": "T06", "decision_id": "add_on", "timestamp": "2026-09-01 10:03:00", "stage": "Browsing", "choice": "none"},
        {"user_id": "T06", "decision_id": "add_on", "timestamp": "2026-09-01 10:05:00", "stage": "Selection", "choice": "analytics"},
    ])
    revs_6 = detect_reversals(load_and_validate(df_chronic, CONFIG), CONFIG)
    m_6 = compute_metrics(load_and_validate(df_chronic, CONFIG), revs_6, CONFIG)
    assert m_6["decisions_with_reversals"].iloc[0] == 2
    assert m_6["max_share_on_one_decision"].iloc[0] == 0.50  # 1/2 on plan, 1/2 on add_on -> broad!
    c_6 = classify_all_users(m_6, revs_6, CONFIG)
    assert c_6["pattern"].iloc[0] == PATTERN_CHRONIC_INDECISIVE
    print("  [6/8] test_chronic_indecision PASSED")
    
    # Test 7: User with zero reversals safe metrics
    m_zero = compute_metrics(load_and_validate(df_no_change, CONFIG), pd.DataFrame(), CONFIG)
    assert m_zero["total_reversals"].iloc[0] == 0
    assert m_zero["reversal_rate"].iloc[0] == 0.0
    assert math.isnan(m_zero["avg_hesitation_seconds"].iloc[0])
    assert m_zero["most_common_reversal_stage"].iloc[0] == ""
    print("  [7/8] test_zero_reversals_metrics PASSED")
    
    # Test 8: Empty input handling
    empty_df = pd.DataFrame(columns=CONFIG["CSV_COLUMNS"])
    revs_empty = detect_reversals(empty_df, CONFIG)
    assert revs_empty.empty
    m_empty = compute_metrics(empty_df, revs_empty, CONFIG)
    assert m_empty.empty
    print("  [8/8] test_empty_input PASSED")
    
    print("\n=======================================================")
    print("All tests passed! (8/8 test suites verified)")
    print("=======================================================")

run_unit_tests()

Running inline assert tests...
  [1/8] test_no_change PASSED
  [2/8] test_single_correction PASSED
  [3/8] test_flip_back PASSED
  [4/8] test_linear_progression PASSED
  [5/8] test_late_reverser PASSED
  [6/8] test_chronic_indecision PASSED
  [7/8] test_zero_reversals_metrics PASSED
  [8/8] test_empty_input PASSED

All tests passed! (8/8 test suites verified)


## Section 11: Benchmark Evaluation & Stress Testing
We evaluate the complete detector against:
1. **Primary Clean Dataset (`SEED=42`)**: Benchmark 200 users across 5 balanced personas.
2. **Second Noisy Dataset (`SEED=999`, Noise Rate = 8%)**: Stress-testing classification robustness when users exhibit stochastic out-of-order interactions.

In [12]:
def evaluate_dataset(
    events_df: pd.DataFrame,
    gt_df: pd.DataFrame,
    dataset_name: str,
    config: Dict[str, Any] = CONFIG,
) -> float:
    """Runs pipeline and outputs accuracy, confusion matrix, and classification report."""
    cleaned = load_and_validate(events_df, config)
    revs = detect_reversals(cleaned, config)
    metrics = compute_metrics(cleaned, revs, config)
    classified = classify_all_users(metrics, revs, config)
    
    merged = pd.merge(gt_df, classified[["user_id", "pattern"]], on="user_id")
    y_true = merged["persona"]
    y_pred = merged["pattern"]
    
    acc = accuracy_score(y_true, y_pred)
    print(f"\n{'=' * 65}")
    print(f"  Evaluation Results: {dataset_name}")
    print(f"{'=' * 65}")
    print(f"Overall Accuracy: {acc * 100:.2f}% ({sum(y_true == y_pred)} / {len(y_true)} users correct)")
    
    # Confusion Matrix
    print(f"\nConfusion Matrix (Rows: True Persona, Cols: Predicted Pattern):")
    cm = pd.crosstab(y_true, y_pred, rownames=["True Persona"], colnames=["Predicted Pattern"])
    display(cm)
    
    # Classification Report
    print(f"\nPer-Class Precision, Recall, and F1-Score:")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))
    return acc

# 1. Primary Dataset Evaluation
acc_primary = evaluate_dataset(clean_events_df, ground_truth_df, "Primary Benchmark (Seed 42)")

# 2. Second Noisy Dataset Evaluation
print("\nGenerating Second Noisy Dataset (Seed 999, Noise Rate 8%)...")
events_noisy, gt_noisy = generate_synthetic_dataset(num_users=200, seed=999, noise_rate=0.08, config=CONFIG)
acc_noisy = evaluate_dataset(events_noisy, gt_noisy, "Secondary Noisy Dataset (Seed 999, Noise=0.08)")

print(f"\nSummary Comparison:")
print(f"  Primary Clean Benchmark Accuracy : {acc_primary * 100:.2f}%")
print(f"  Secondary Noisy Benchmark Accuracy: {acc_noisy * 100:.2f}%")


  Evaluation Results: Primary Benchmark (Seed 42)
Overall Accuracy: 100.00% (200 / 200 users correct)

Confusion Matrix (Rows: True Persona, Cols: Predicted Pattern):


Predicted Pattern,chronic_indecisive,decisive,flip_flopper,late_reverser,single_corrector
True Persona,,,,,
chronic_indecisive,40,0,0,0,0
decisive,0,40,0,0,0
flip_flopper,0,0,40,0,0
late_reverser,0,0,0,40,0
single_corrector,0,0,0,0,40



Per-Class Precision, Recall, and F1-Score:
                    precision    recall  f1-score   support

chronic_indecisive     1.0000    1.0000    1.0000        40
          decisive     1.0000    1.0000    1.0000        40
      flip_flopper     1.0000    1.0000    1.0000        40
     late_reverser     1.0000    1.0000    1.0000        40
  single_corrector     1.0000    1.0000    1.0000        40

          accuracy                         1.0000       200
         macro avg     1.0000    1.0000    1.0000       200
      weighted avg     1.0000    1.0000    1.0000       200


Generating Second Noisy Dataset (Seed 999, Noise Rate 8%)...



  Evaluation Results: Secondary Noisy Dataset (Seed 999, Noise=0.08)
Overall Accuracy: 95.00% (190 / 200 users correct)

Confusion Matrix (Rows: True Persona, Cols: Predicted Pattern):


Predicted Pattern,chronic_indecisive,decisive,flip_flopper,late_reverser,mixed,single_corrector
True Persona,,,,,,
chronic_indecisive,40,0,0,0,0,0
decisive,0,37,0,2,0,1
flip_flopper,0,0,40,0,0,0
late_reverser,0,0,0,39,1,0
single_corrector,0,0,0,0,6,34



Per-Class Precision, Recall, and F1-Score:
                    precision    recall  f1-score   support

chronic_indecisive     1.0000    1.0000    1.0000        40
          decisive     1.0000    0.9250    0.9610        40
      flip_flopper     1.0000    1.0000    1.0000        40
     late_reverser     0.9512    0.9750    0.9630        40
             mixed     0.0000    0.0000    0.0000         0
  single_corrector     0.9714    0.8500    0.9067        40

          accuracy                         0.9500       200
         macro avg     0.8204    0.7917    0.8051       200
      weighted avg     0.9845    0.9500    0.9661       200


Summary Comparison:
  Primary Clean Benchmark Accuracy : 100.00%
  Secondary Noisy Benchmark Accuracy: 95.00%


### ⚠️ Honest Reflection on Synthetic Data & Real-World Limitations

1. **Synthetic Bias**: The benchmark personas are generated via synthetic simulation. Real human behavior features overlapping motivations, external distractions (e.g. browser tab timeouts), multi-session purchasing spanning days, and intermittent network retries.
2. **Deterministic Boundaries**: The rule priority (`decisive → single_corrector → chronic_indecisive → flip_flopper → late_reverser → mixed`) assumes crisp thresholds (e.g. $\le 60\text{s}$, $60\%$ share). In production, these should be continuously calibrated using conversion outcome telemetry.
3. **Session Context**: The current implementation treats each user's event log as a single continuous session. Extending to multi-session attribution will allow tracking hesitation decay over time.

## Section 12: Optional Affective Layer (Behavioral Uncertainty)
*Controlled by `CONFIG["ENABLE_AFFECTIVE_LAYER"]`.*

> **Important Conceptual Note**: This module does **not** perform facial emotion recognition or sentiment analysis. Rather, it computes a mathematical **behavioral uncertainty proxy score** in $[0, 1]$ and fits a 2-state Gaussian Hidden Markov Model (HMM) over hesitation deltas ($\Delta t$):
> - **State 0 (Confident)**: Low hesitation, steady linear progress.
> - **State 1 (Hesitant / Oscillating)**: Elevated hesitation variance and choice oscillation.

$$\text{Uncertainty Score} = 0.45 \times \min\left(1.0, \frac{\text{reversal\_rate}}{2.0}\right) + 0.35 \times \frac{\text{flip\_backs}}{\max(1, \text{reversals})} + 0.20 \times \min\left(1.0, \frac{\text{avg\_hesitation}}{120\text{s}}\right)$$

In [13]:
from hmmlearn import hmm

def compute_uncertainty_score(row: pd.Series) -> float:
    """Computes a continuous behavioral uncertainty proxy score in [0.0, 1.0]."""
    n_revs = row["total_reversals"]
    if n_revs == 0:
        return 0.0
    
    # Factor 1: Reversal rate (normalized, 2.0 = ceiling)
    r_factor = min(1.0, row["reversal_rate"] / 2.0)
    
    # Factor 2: Flip-back ratio
    f_factor = row["flip_back_count"] / max(1, n_revs)
    
    # Factor 3: Hesitation duration (normalized, 120s = ceiling)
    h_val = row["avg_hesitation_seconds"]
    h_factor = min(1.0, h_val / 120.0) if not math.isnan(h_val) else 0.0
    
    score = (0.45 * r_factor) + (0.35 * f_factor) + (0.20 * h_factor)
    return round(float(np.clip(score, 0.0, 1.0)), 4)

if CONFIG["ENABLE_AFFECTIVE_LAYER"]:
    print("🧠 Affective Layer ENABLED. Fitting 2-State Gaussian HMM on hesitation intervals...")
    
    # Attach uncertainty score
    explained_df["uncertainty_score"] = explained_df.apply(compute_uncertainty_score, axis=1)
    
    # Train 2-State Gaussian HMM on all detected hesitation intervals
    hesitations = reversals_df["seconds_since_previous"].values.reshape(-1, 1)
    if len(hesitations) >= 10:
        hmm_model = hmm.GaussianHMM(n_components=2, covariance_type="diag", n_iter=100, random_state=CONFIG["SEED"])
        hmm_model.fit(hesitations)
        
        # Sort states: State 0 = low hesitation (Confident), State 1 = high hesitation (Uncertain)
        means = hmm_model.means_.flatten()
        confident_state = int(np.argmin(means))
        uncertain_state = int(np.argmax(means))
        
        print(f"HMM Convergence: Confident Mean = {means[confident_state]:.1f}s | Uncertain Mean = {means[uncertain_state]:.1f}s")
        
        # Plot state strip chart for sample oscillating user
        sample_u = explained_df[explained_df["pattern"] == PATTERN_FLIP_FLOPPER].iloc[0]["user_id"]
        u_hes = reversals_df[reversals_df["user_id"] == sample_u]["seconds_since_previous"].values.reshape(-1, 1)
        if len(u_hes) > 0:
            hidden_states = hmm_model.predict(u_hes)
            state_labels = ["Confident" if s == confident_state else "Uncertain" for s in hidden_states]
            
            fig_hmm = go.Figure(go.Scatter(
                y=[f"Reversal #{i+1}" for i in range(len(state_labels))],
                x=u_hes.flatten(),
                mode="markers+text",
                marker=dict(size=14, color=["#10b981" if s == "Confident" else "#ef4444" for s in state_labels]),
                text=state_labels,
                textposition="middle right",
            ))
            fig_hmm.update_layout(
                title=f"Affective State Sequence — User {sample_u}",
                xaxis_title="Hesitation Interval (seconds)",
                template="plotly_dark",
                height=280
            )
            fig_hmm.show()
    display(explained_df[["user_id", "pattern", "uncertainty_score"]].head())
else:
    print("ℹ️ Section 12 Affective Layer is currently DISABLED in CONFIG.")
    print("   To enable behavioral uncertainty scoring & Gaussian HMM inference, set:")
    print("   CONFIG['ENABLE_AFFECTIVE_LAYER'] = True and re-run.")

ℹ️ Section 12 Affective Layer is currently DISABLED in CONFIG.
   To enable behavioral uncertainty scoring & Gaussian HMM inference, set:
   CONFIG['ENABLE_AFFECTIVE_LAYER'] = True and re-run.


## Section 13: Export & Executive Summary
Writes all structured analytical artifacts to `outputs/` and outputs the final executive summary.

In [14]:
os.makedirs("outputs", exist_ok=True)

# 1. Export user metrics, classifications, and explanations
export_metrics_path = "outputs/user_metrics_and_patterns.csv"
explained_df.to_csv(export_metrics_path, index=False)

# 2. Export detected reversal event records
export_revs_path = "outputs/detected_reversals.csv"
reversals_df.to_csv(export_revs_path, index=False)

print(f"✅ Exported: {export_metrics_path} ({len(explained_df)} users)")
print(f"✅ Exported: {export_revs_path} ({len(reversals_df)} reversal events)")

print("\n" + "=" * 70)
print("                    FINAL EXECUTIVE SUMMARY")
print("=" * 70)
print(f"Total Users Evaluated           : {len(explained_df):,}")
print(f"Total Events Ingested           : {len(clean_events_df):,}")
print(f"Total Reversals Detected        : {len(reversals_df):,}")
print(f"Flip-Backs (Circular Regret)    : {reversals_df['is_flip_back'].sum():,}")
print(f"Primary Benchmark Accuracy      : {acc_primary * 100:.2f}% (200 / 200 correct)")
print("-" * 70)
print("Classification Breakdown:")
for pat, cnt in explained_df["pattern"].value_counts().items():
    pct = (cnt / len(explained_df)) * 100
    bar = "█" * int(pct / 2.5)
    print(f"  {pat:<20} : {cnt:>4} ({pct:>5.1f}%)  {bar}")

print("-" * 70)
print("Top 5 Most Reversal-Prone Users (High Churn / Oscillations):")
top_prone = explained_df.sort_values(by=["reversal_rate", "total_reversals"], ascending=False).head(5)
for _, r in top_prone.iterrows():
    print(f"  • {r['user_id']} ({r['pattern']}): {r['total_reversals']} reversals | rate: {r['reversal_rate']:.2f} | flips: {r['flip_back_count']}")
    print(f"    Evidence: {r['explanation']}\n")
print("=" * 70)
print("CEREBRO Notebook Pipeline Execution Complete.")

✅ Exported: outputs/user_metrics_and_patterns.csv (200 users)
✅ Exported: outputs/detected_reversals.csv (639 reversal events)

                    FINAL EXECUTIVE SUMMARY
Total Users Evaluated           : 200
Total Events Ingested           : 1,239
Total Reversals Detected        : 639
Flip-Backs (Circular Regret)    : 223
Primary Benchmark Accuracy      : 100.00% (200 / 200 correct)
----------------------------------------------------------------------
Classification Breakdown:
  single_corrector     :   40 ( 20.0%)  ████████
  chronic_indecisive   :   40 ( 20.0%)  ████████
  flip_flopper         :   40 ( 20.0%)  ████████
  late_reverser        :   40 ( 20.0%)  ████████
  decisive             :   40 ( 20.0%)  ████████
----------------------------------------------------------------------
Top 5 Most Reversal-Prone Users (High Churn / Oscillations):
  • U0016 (chronic_indecisive): 11 reversals | rate: 3.67 | flips: 4
    Evidence: U0016 changed decisions broadly across 3 area(s) with a